# Timeseries management: from readings to a queryable table

A BTwin graph says a building *has* a temperature sensor in the Open Office. It does not say what
that sensor read at 14:15 last Tuesday. Readings live somewhere else — in this library, in a
SQLite table shaped by `Observation.Template()` — and this notebook is about getting them in,
getting them out, and describing the table well enough that something other than you can query it.

Nothing here calls a model. Everything runs offline, and the last three sections build exactly the
grounding and the safety rails that [tutorial 06](../06-chat-with-timeseries/chat-with-timeseries.ipynb)
hands to one.

The route:

1. the sensors, as BTwin points
2. the observation shape, and a week of synthetic readings
3. into SQLite
4. out again — the typed API, then raw SQL
5. describing the table for something that has never seen it
6. the gate a generated query has to pass
7. changing the table without being able to lose it

In [1]:
import random
from pathlib import Path

import pandas as pd

import btwin
from btwin import SQL, Observation, Point, SpatialElement

OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)

DB = OUTPUT / "office.db"
TABLE = "observations"

print("BTwin", btwin.__version__)

BTwin 0.5.7


## 1. The sensors

Ordinary BTwin objects, from tutorial 01: a space, and four points that measure something about
it. This is the part a graph holds — identity, type, where the thing is.

In [2]:
office = SpatialElement.Constructor("space-01", "bot:Space", "Open Office")

sensors = {
    "OFF-TEMP-01":  ("brick:Temperature_Sensor", "Open Office Temperature"),
    "OFF-CO2-01":   ("brick:CO2_Sensor",         "Open Office CO2"),
    "OFF-POWER-01": ("brick:Electric_Power_Sensor", "Open Office Power"),
    "OFF-OCC-01":   ("brick:Occupancy_Sensor",   "Open Office Occupancy"),
}

points = []
for uid, (pointType, name) in sensors.items():
    point = Point.Constructor(uid, pointType, name)
    Point.SetRelationship(point, "brick:hasLocation", linkedObject=office)
    points.append(point)

for point in points:
    print(f"{Point.UID(point):<14} {point['@type']:<30} {Point.Name(point)}")

OFF-TEMP-01    brick:Temperature_Sensor       Open Office Temperature
OFF-CO2-01     brick:CO2_Sensor               Open Office CO2
OFF-POWER-01   brick:Electric_Power_Sensor    Open Office Power
OFF-OCC-01     brick:Occupancy_Sensor         Open Office Occupancy


## 2. The observation shape

`Observation.Template()` is the contract: five columns, one row per reading. Everything else in
this notebook — the query API, the grounding block, the validator — assumes this shape.

In [3]:
Observation.Template()

,sosa:madeBySensor,sosa:ObservedProperty,unit,value,timestamp
0,temperaturePoint1,Temperature,°C,22.5,2025-03-01 10:00
1,temperaturePoint1,Temperature,°C,22.7,2025-03-01 10:10
2,temperaturePoint1,Temperature,°C,22.8,2025-03-01 10:20
3,temperaturePoint1,Temperature,°C,22.3,2025-03-01 10:30


The column names are SOSA's, and two of them carry a colon. That is not decoration: it is why
`"sosa:madeBySensor"` has to be double-quoted in every hand-written query below, and why the
grounding block in section 5 spends a line saying so.

## 3. A week of readings

One office, four sensors, 15-minute intervals, Monday 3 March to Sunday 9 March 2025. Seeded, so
the numbers below are the numbers you will get.

The profile is the point of the exercise: occupancy drives CO2 and power, the weekend empties the
building, and temperature is held at a setpoint while anyone is in it and drifts when nobody is.
An aggregate that cannot see that difference is measuring the wrong thing.

In [4]:
STEP_MINUTES = 15
DAYS = 7
START = pd.Timestamp("2025-03-03T00:00:00Z")   # a Monday

rng = random.Random(20260901)
rows = []

for step in range(DAYS * 24 * 60 // STEP_MINUTES):
    stamp = START + pd.Timedelta(minutes=STEP_MINUTES * step)
    hour = stamp.hour + stamp.minute / 60
    weekend = stamp.dayofweek >= 5

    # Occupancy: nobody at the weekend, a ramp in and out on a working day
    if weekend or hour < 7.5 or hour > 18.5:
        occupancy = 0
    elif hour < 9:
        occupancy = round((hour - 7.5) / 1.5 * 22 * rng.uniform(0.8, 1.2))
    elif hour > 17:
        occupancy = round((18.5 - hour) / 1.5 * 22 * rng.uniform(0.8, 1.2))
    else:
        occupancy = round(22 * rng.uniform(0.75, 1.1))

    # CO2 follows the people in the room, over an outdoor baseline
    co2 = 420 + occupancy * 24 * rng.uniform(0.9, 1.1)
    # Power: a standing load, plus workstations and lighting while occupied
    power = 4.2 + occupancy * 0.28 * rng.uniform(0.9, 1.1) + (2.6 if occupancy else 0)
    # Temperature: held at setpoint when occupied, drifting down overnight
    temperature = (21.0 + rng.uniform(-0.4, 0.4)) if occupancy else (18.4 + rng.uniform(-0.8, 0.8))

    iso = stamp.strftime("%Y-%m-%dT%H:%M:%SZ")
    rows += [
        ["OFF-TEMP-01",  "Temperature",      "degC",   round(temperature, 2), iso],
        ["OFF-CO2-01",   "CO2Concentration", "ppm",    round(co2, 1),         iso],
        ["OFF-POWER-01", "ElectricalPower",  "kW",     round(power, 3),       iso],
        ["OFF-OCC-01",   "Occupancy",        "people", float(occupancy),      iso],
    ]

observations = pd.DataFrame(rows, columns=list(Observation.Template().columns))
print(f"{len(observations)} rows, {observations['sosa:madeBySensor'].nunique()} sensors, "
      f"{observations['timestamp'].nunique()} timestamps")
observations.head()

2688 rows, 4 sensors, 672 timestamps


,sosa:madeBySensor,sosa:ObservedProperty,unit,value,timestamp
0,OFF-TEMP-01,Temperature,degC,18.41,2025-03-03T00:00:00Z
1,OFF-CO2-01,CO2Concentration,ppm,420.00,2025-03-03T00:00:00Z
2,OFF-POWER-01,ElectricalPower,kW,4.20,2025-03-03T00:00:00Z
3,OFF-OCC-01,Occupancy,people,0.00,2025-03-03T00:00:00Z
4,OFF-TEMP-01,Temperature,degC,18.55,2025-03-03T00:15:00Z


## 4. Into SQLite

`Observation.SQLiteByDF` writes the frame to a table. `ifExists="replace"` makes the notebook
re-runnable; `"append"` is what a nightly ingest would use, and `"fail"` — the default — refuses
to touch a table that is already there.

In [5]:
path = Observation.SQLiteByDF(observations, str(DB), TABLE, ifExists="replace")
print(path)
print(f"{DB.stat().st_size / 1024:.0f} KB on disk")

C:\Users\massa\OneDrive - Alma Mater Studiorum Università di Bologna\Desktop\Scripts\btwin\btwinpy-00.05.03\tutorials\05-timeseries-management\output\office.db
176 KB on disk


`Observation.SQLiteByXLSX` does the same from a spreadsheet, which is how bills and meter exports
usually arrive. It reads with pandas and hands straight over to `SQLiteByDF`, so everything below
applies equally to a table that came from Excel.

## 5. Out again: the typed API

`Observation.SQLiteQuery` builds the SQL for you. You say which sensor, which aggregate, which
period — it never sees a query you wrote, so there is nothing to validate.

In [6]:
# Every reading from one sensor, for one morning
Observation.SQLiteQuery(
    str(DB), TABLE,
    sensor="OFF-CO2-01",
    startTime="2025-03-04T08:00:00Z",
    endTime="2025-03-04T09:30:00Z",
)

,sosa:madeBySensor,sosa:ObservedProperty,unit,value,timestamp
0,OFF-CO2-01,CO2Concentration,ppm,593.6,2025-03-04T08:00:00Z
1,OFF-CO2-01,CO2Concentration,ppm,706.0,2025-03-04T08:15:00Z
2,OFF-CO2-01,CO2Concentration,ppm,720.6,2025-03-04T08:30:00Z
3,OFF-CO2-01,CO2Concentration,ppm,848.9,2025-03-04T08:45:00Z
4,OFF-CO2-01,CO2Concentration,ppm,946.3,2025-03-04T09:00:00Z
5,OFF-CO2-01,CO2Concentration,ppm,888.0,2025-03-04T09:15:00Z
6,OFF-CO2-01,CO2Concentration,ppm,859.8,2025-03-04T09:30:00Z


In [7]:
# Daily peaks, per sensor: the profile in section 3, read back out
Observation.SQLiteQuery(str(DB), TABLE, aggregate="max", groupByTime="day")

,sosa:madeBySensor,sosa:ObservedProperty,unit,period,Value_max
0,OFF-OCC-01,Occupancy,people,2025-03-03,24.000
1,OFF-TEMP-01,Temperature,degC,2025-03-03,21.390
2,OFF-CO2-01,CO2Concentration,ppm,2025-03-03,1010.300
3,OFF-POWER-01,ElectricalPower,kW,2025-03-03,14.152
4,OFF-OCC-01,Occupancy,people,2025-03-04,24.000
5,OFF-TEMP-01,Temperature,degC,2025-03-04,21.350
6,OFF-POWER-01,ElectricalPower,kW,2025-03-04,13.816
7,OFF-CO2-01,CO2Concentration,ppm,2025-03-04,1019.900
8,OFF-TEMP-01,Temperature,degC,2025-03-05,21.380
9,OFF-CO2-01,CO2Concentration,ppm,2025-03-05,1010.000


Read the `Occupancy` rows down that table and the weekend is right there: 0 on the 8th and the
9th, in the low twenties every working day.

`aggregate` takes `min`, `max`, `mean`, `sum` and `count`; `groupByTime` takes `hour`, `day` and
`month`. Between them they cover most of what a dashboard asks for, and none of it needs a line of
SQL.

## 6. Out again: raw SQL

When the question does not fit those parameters — a comparison between two periods, a filter on a
derived value, anything with a `CASE` — you write the SQL yourself and `Observation.SQLiteFetch`
runs it.

In [8]:
occupiedHours = Observation.SQLiteFetch(str(DB), '''
    SELECT strftime('%Y-%m-%d', timestamp) AS day,
           ROUND(AVG(value), 1) AS meanCO2
    FROM observations
    WHERE "sosa:ObservedProperty" = 'CO2Concentration'
      AND CAST(strftime('%H', timestamp) AS INTEGER) BETWEEN 9 AND 17
    GROUP BY day
    ORDER BY day
''')
pd.DataFrame(occupiedHours)

,day,meanCO2
0,2025-03-03,876.4
1,2025-03-04,901.2
2,2025-03-05,897.5
3,2025-03-06,885.8
4,2025-03-07,910.7
5,2025-03-08,420.0
6,2025-03-09,420.0


Note the double quotes around `"sosa:ObservedProperty"` — without them SQLite reads the colon as
the start of a parameter marker and the query will not compile.

The connection is opened in SQLite's **read-only** mode, so `SQLiteFetch` cannot change the file
whatever it is handed:

In [9]:
import sqlite3

try:
    Observation.SQLiteFetch(str(DB), "DELETE FROM observations")
except sqlite3.DatabaseError as exc:
    print("refused:", exc)

print("still", Observation.SQLiteFetch(str(DB), "SELECT COUNT(*) AS n FROM observations")[0]["n"], "rows")

refused: SQLite query failed: attempt to write a readonly database
still 2688 rows


## 7. Describing the table

Everything so far assumed you already know what is in the table. Something that does not — a
colleague, a dashboard, a model — needs to be told, and `Observation.SQLiteIndex` is what reads it
off the data.

In [10]:
index = Observation.SQLiteIndex(str(DB), TABLE)

print(f"{index['rows']} rows\n")
for column in index["columns"]:
    listed = ", ".join(repr(v) for v in column["values"]) if column["values"] else "(not listed)"
    print(f"{column['name']:<24} {column['type']:<6} {column['distinct']:>4} distinct   {listed}")

2688 rows

sosa:madeBySensor        TEXT      4 distinct   'OFF-CO2-01', 'OFF-OCC-01', 'OFF-POWER-01', 'OFF-TEMP-01'
sosa:ObservedProperty    TEXT      4 distinct   'CO2Concentration', 'ElectricalPower', 'Occupancy', 'Temperature'
unit                     TEXT      4 distinct   'degC', 'kW', 'people', 'ppm'
value                    REAL    661 distinct   (not listed)
timestamp                TEXT    672 distinct   (not listed)


Three columns are listed in full and two are not, and the rule behind that is worth being explicit
about:

- **`unit` and the two SOSA columns are vocabulary.** A question names a sensor or a quantity in
  words; the table stores them as opaque strings, and nothing in a SQL schema says which strings
  exist. Anybody writing a `WHERE` clause has to know that the spelling is `CO2Concentration` and
  not `CO2` — so the complete list goes in.
- **`value` and `timestamp` are not.** 672 timestamps would drown the description in the one thing
  anybody can already reason about, and listing readings as if they were legal values invites a
  filter on them. Both are given as a range instead.

`Observation.SQLiteSchemaSummary` renders that as a block of text:

In [11]:
schema = Observation.SQLiteSchemaSummary(
    str(DB), TABLE, index,
    notes="A sensor identifier reads SPACE-QUANTITY-INDEX, so every sensor in this table is in "
          "the Open Office. Occupancy is a count of people, not a percentage.",
)
print(schema["text"])

TABLE
  "observations"  (2688 rows)

COLUMNS (name, SQLite type, what it spans)
  "sosa:madeBySensor"        TEXT      4 distinct, from 'OFF-CO2-01' to 'OFF-TEMP-01'
  "sosa:ObservedProperty"    TEXT      4 distinct, from 'CO2Concentration' to 'Temperature'
  "unit"                     TEXT      4 distinct, from 'degC' to 'ppm'
  "value"                    REAL      661 distinct, from 0.0 to 1047.0
  "timestamp"                TEXT      672 distinct, from '2025-03-03T00:00:00Z' to '2025-03-09T23:45:00Z'

VALUES (the complete contents of the columns short enough to list -
        match these exactly, they are the only ones in the table)
  "sosa:madeBySensor": 'OFF-CO2-01', 'OFF-OCC-01', 'OFF-POWER-01', 'OFF-TEMP-01'
  "sosa:ObservedProperty": 'CO2Concentration', 'ElectricalPower', 'Occupancy', 'Temperature'
  "unit": 'degC', 'kW', 'people', 'ppm'

NOTES
  A column name containing ':' or a space MUST be double-quoted, e.g.
    SELECT "sosa:madeBySensor" FROM "observations"
  "timestamp" 

The NOTES section is half generated and half yours. The generated half is SQLite dialect — the
quoting rule, and the fact that `timestamp` is ISO 8601 **text** rather than a date type, so it is
read with `strftime` and there is no `EXTRACT` or `DATE_TRUNC` to reach for.

The `notes` half is the part the table cannot say about itself. A graph carries labels and types;
a flat table carries neither, so what `OFF-CO2-01` *means* is known only to you. Section 5 of
[tutorial 06](../06-chat-with-timeseries/chat-with-timeseries.ipynb) shows what happens when that
is left out.

## 8. The gate

A query written by hand is your problem. A query written by anything else has to be checked before
it reaches the database, and `SQL.Validate` is that check: shape, safety, one statement,
compilation, limit.

In [12]:
candidates = [
    'SELECT "unit", AVG(value) FROM observations GROUP BY "unit"',
    'SELECT AVG(temperature) FROM observations',
    'DROP TABLE observations',
    'SELECT value FROM observations; DELETE FROM observations',
    "SELECT replace(\"unit\", 'degC', 'K') FROM observations LIMIT 5",
]

for candidate in candidates:
    checked, error = SQL.Validate(candidate, str(DB), rowLimit=100, columns=schema["columns"])
    print(f"{'PASS' if checked else 'STOP'}  {candidate[:52]:<54} {error[:64]}")

PASS  SELECT "unit", AVG(value) FROM observations GROUP BY   
STOP  SELECT AVG(temperature) FROM observations              SQLite rejected the query: no such column: temperature. Columns 
STOP  DROP TABLE observations                                DROP is not allowed; write a SELECT (a WITH ... SELECT is fine).
STOP  SELECT value FROM observations; DELETE FROM observat   'DELETE' is not allowed in this query.
PASS  SELECT replace("unit", 'degC', 'K') FROM observation   


The second line is the one that earns the whole exercise. `temperature` is a plausible column name
and there is no such column — but a query that merely *parses* would run, return nothing, and read
exactly like an honest "no data". SQLite is asked to `EXPLAIN` the query instead, which compiles it
in full, resolving every table, column and function, **without executing a row**. The hallucinated
name is refused by name.

The first line came back `PASS` and was rewritten: a missing `LIMIT` is a defect the validator
repairs rather than a reason to reject.

In [13]:
checked, _ = SQL.Validate('SELECT "unit", AVG(value) FROM observations GROUP BY "unit"',
                          str(DB), rowLimit=100)
print(checked)

SELECT "unit", AVG(value) FROM observations GROUP BY "unit"
LIMIT 100


`SQL.ValidateUpdate` is the same gate with the shape test inverted — `INSERT`, `UPDATE` and
`DELETE` are the only openings it accepts. Three of its refusals are worth seeing, because each is
a way of destroying data while answering the request exactly as put.

In [14]:
writes = [
    'UPDATE observations SET value = 0 WHERE "sosa:madeBySensor" = \'OFF-OCC-01\'',
    'DELETE FROM observations',
    'UPDATE observations SET value = 0',
    "REPLACE INTO observations VALUES ('a', 'b', 'c', 1.0, 'd')",
    'INSERT INTO otherTable VALUES (1)',
]

for candidate in writes:
    checked, error = SQL.ValidateUpdate(candidate, TABLE, str(DB), schema["columns"])
    print(f"{'PASS' if checked else 'STOP'}  {candidate[:46]:<48} {error[:70]}")

PASS  UPDATE observations SET value = 0 WHERE "sosa:   
STOP  DELETE FROM observations                         A DELETE must carry a WHERE clause naming the rows to change. Without 
STOP  UPDATE observations SET value = 0                An UPDATE must carry a WHERE clause naming the rows to change. Without
STOP  REPLACE INTO observations VALUES ('a', 'b', 'c   REPLACE deletes the row it collides with. Write a plain INSERT to add 
STOP  INSERT INTO otherTable VALUES (1)                This update writes to 'otherTable'. It may only write to 'observations


- **No `WHERE`** empties or rewrites the whole table. It is the table's version of the `DROP` and
  `CLEAR` that `SPARQL.ValidateUpdate` refuses outright on the graph side.
- **`REPLACE`** deletes whatever row it collides with, so a statement that reads as an addition
  silently removes data nobody mentioned.
- **Another table** is out of scope by construction — the counterpart of confining a SPARQL update
  to the default graph.

## 9. Changing the table

Say the occupancy sensor was miscalibrated on the Wednesday and every reading that day is 20% low.
The fix is an `UPDATE`, and `Observation.SQLiteApplyUpdate` runs it **inside a transaction** — so
you can read what it did before deciding whether to keep it.

In [15]:
correction = '''
    UPDATE observations
    SET value = ROUND(value * 1.25)
    WHERE "sosa:madeBySensor" = 'OFF-OCC-01'
      AND strftime('%Y-%m-%d', timestamp) = '2025-03-05'
      AND value > 0
'''

changes, added, removed, error = Observation.SQLiteApplyUpdate(str(DB), TABLE, correction)
print(f"{changes} row(s) would change, error={error!r}\n")

# 'added' and 'removed' are two multisets of whole rows, NOT a paired before/after: nothing in
# the API knows that this row became that one. Pairing them on the timestamp is a judgement
# about this particular UPDATE, which touched one row per interval and left the key alone.
before = {row["timestamp"]: row["value"] for row in removed}
after = {row["timestamp"]: row["value"] for row in added}
for stamp in sorted(before)[:5]:
    print(f"  {stamp}   {before[stamp]:>5}  ->  {after[stamp]:>5}")

43 row(s) would change, error=''

  2025-03-05T07:45:00Z     4.0  ->    5.0
  2025-03-05T08:00:00Z     6.0  ->    8.0
  2025-03-05T08:15:00Z     9.0  ->   11.0
  2025-03-05T08:30:00Z    12.0  ->   15.0
  2025-03-05T08:45:00Z    18.0  ->   23.0


In [16]:
# ...and the file has not moved. commit defaults to False, so the transaction was rolled back.
Observation.SQLiteFetch(str(DB), '''
    SELECT MAX(value) AS peak FROM observations
    WHERE "sosa:madeBySensor" = 'OFF-OCC-01'
      AND strftime('%Y-%m-%d', timestamp) = '2025-03-05'
''')

[{'peak': 24.0}]

That is the whole idea: a rehearsal you can read, and a file that has not changed.

Two properties of that diff are worth being precise about, because both bite:

- It is a **multiset** difference, not a set difference. That matters here — two 15-minute
  intervals can legitimately record the same occupancy at the same value, and a set diff would
  report correcting one of them as no change at all.
- It is **two lists, not a mapping**. `removed` is what disappeared and `added` is what appeared;
  nothing pairs them up, because in general nothing can — an `UPDATE` that rewrote a key, or a
  statement that deletes some rows and inserts others, has no row-to-row correspondence to
  report. The cell above pairs them on the timestamp because *this* statement leaves the
  timestamp alone, and that is the caller's knowledge, not the API's.

To keep it, commit — to a **copy**, so the readings you started from are still there to compare
against:

In [17]:
edited = Observation.SQLiteCopy(str(DB), str(OUTPUT / "office_corrected.db"))
changes, _, _, _ = Observation.SQLiteApplyUpdate(edited, TABLE, correction, commit=True)

peak = 'SELECT MAX(value) AS peak FROM observations WHERE "sosa:madeBySensor" = \'OFF-OCC-01\' AND strftime(\'%Y-%m-%d\', timestamp) = \'2025-03-05\''
print(f"{changes} row(s) committed")
print("original :", Observation.SQLiteFetch(str(DB), peak)[0]["peak"])
print("corrected:", Observation.SQLiteFetch(edited, peak)[0]["peak"])

43 row(s) committed
original : 24.0
corrected: 30.0


`Observation.SQLiteCopy` goes through SQLite's own backup API rather than copying bytes: a database
with a write-ahead log lives in more than one file, and copying the main one gives a snapshot
missing its most recent commits.

## What you have

`output/office.db` — a week of readings, untouched — and `output/office_corrected.db`, the same
week with Wednesday's occupancy fixed. Two files that can be diffed, which is the only reason to
prefer a copy over an in-place edit.

And, less visibly, everything a model needs to work on this table without being able to break it:

| | |
|---|---|
| `Observation.SQLiteIndex` / `SQLiteSchemaSummary` | what is in the table, in words |
| `SQL.Validate` / `SQL.ValidateUpdate` | what a generated statement is allowed to be |
| `Observation.SQLiteFetch` | reads, on a connection that cannot write |
| `Observation.SQLiteApplyUpdate` | writes, rehearsed before they are kept |

[Tutorial 06](../06-chat-with-timeseries/chat-with-timeseries.ipynb) hands all four to one and asks
it questions in English.